# Tugas Praktikum Pertemuan 3 — Ekstraksi Fitur

**Mata kuliah:** Pembelajaran Mesin  
**Nama:** Naufal Ramadhan  
**NIM:** 244107020201  

## Dataset Wisconsin Breast Cancer

Tujuan praktikum: memisahkan variabel yang digunakan dan tidak digunakan, melakukan encoding target `diagnosis`, menstandarisasi fitur numerik, dan membuat **stratified split** dengan rasio 80:20.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

DATA_PATH = "file-dataset/js2.csv"
df = pd.read_csv(DATA_PATH)
print(f"Ukuran data: {df.shape}")
df.head()

## 1. Pemeriksaan dan pemisahan variabel

Kolom `id` hanya merupakan identitas observasi sehingga tidak dipakai sebagai fitur prediktif. Kolom `Unnamed: 32` seluruhnya kosong dan juga dibuang. `diagnosis` adalah target; seluruh kolom pengukuran numerik menjadi fitur.

In [ ]:
# Kolom kosong sepenuhnya dibuang, lalu ID dipisahkan dari data analisis
df = df.drop(columns=[c for c in df.columns if df[c].isna().all()])

TARGET = "diagnosis"
UNUSED_COLUMNS = ["id"]
feature_columns = [c for c in df.columns if c not in UNUSED_COLUMNS + [TARGET]]

X_raw = df[feature_columns].copy()
y_raw = df[TARGET].copy()

print("Kolom tidak digunakan:", UNUSED_COLUMNS + ["Unnamed: 32"])
print("Kolom target:", TARGET)
print(f"Jumlah fitur yang digunakan: {len(feature_columns)}")
print("Semua fitur numerik:", X_raw.select_dtypes(include="number").shape[1] == len(feature_columns))
print("Distribusi target asli:")
print(y_raw.value_counts())

## 2. Encoding kolom `diagnosis`

Label encoding mengubah kelas kategorikal menjadi bilangan. Dengan `LabelEncoder`, kelas `B` menjadi `0` dan `M` menjadi `1`. Pemetaan ini dipastikan secara eksplisit agar interpretasi hasil tidak ambigu.

In [ ]:
encoder = LabelEncoder()
y = pd.Series(encoder.fit_transform(y_raw), name=TARGET, index=y_raw.index)
label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print("Pemetaan encoding:", label_mapping)
print("Distribusi target setelah encoding:")
print(y.value_counts().sort_index())

## 3. Standardisasi fitur numerik

Standardisasi dilakukan dengan rumus $z=(x-\mu)/\sigma$. Parameter scaler dihitung dari data latih saja setelah pembagian data, sehingga tidak terjadi **data leakage** dari data uji.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=feature_columns,
    index=X_train_raw.index,
)
X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=feature_columns,
    index=X_test_raw.index,
)

print("Rata-rata fitur train setelah standardisasi (maks absolut):", X_train.mean().abs().max())
print("Simpangan baku fitur train setelah standardisasi (selisih dari 1):", (X_train.std(ddof=0) - 1).abs().max())
print("Contoh fitur terstandarisasi:")
X_train.iloc[:5, :5]

## 4. Stratified split data 80:20

`stratify=y` menjaga proporsi kelas `B` dan `M` pada data latih dan data uji tetap mendekati proporsi dataset awal.

In [ ]:
def distribution_table(labels):
    counts = labels.value_counts().sort_index()
    return pd.DataFrame({"jumlah": counts, "proporsi": (counts / len(labels)).round(4)})

print(f"Data awal : {len(y)} baris")
print(f"Data latih: {len(X_train)} baris ({len(X_train)/len(y):.1%})")
print(f"Data uji  : {len(X_test)} baris ({len(X_test)/len(y):.1%})")

comparison = pd.concat(
    {
        "awal": distribution_table(y),
        "latih": distribution_table(y_train),
        "uji": distribution_table(y_test),
    }, axis=1
)
comparison.index = ["B (0)", "M (1)"]
comparison

## Kesimpulan

1. `id` dan `Unnamed: 32` tidak digunakan; `diagnosis` ditetapkan sebagai target; 30 kolom pengukuran digunakan sebagai fitur.
2. Encoding menghasilkan `B = 0` dan `M = 1`.
3. Seluruh fitur numerik distandarisasi menggunakan parameter yang hanya dipelajari dari data latih.
4. Data dibagi secara stratified menjadi 80% data latih dan 20% data uji, sehingga komposisi kelas tetap terjaga.